# Validation Study of GFM-RAG: Graph Foundation Models for Retrieval-Augmented Generation
**Author:** Eugene Phang  
**Reference Paper:** [GFM-RAG: Graph Foundation Models for Retrieval-Augmented Generation (arXiv:2502.01113)](https://arxiv.org/abs/2502.01113)  
**Repository Reference:** [RManLuo/gfm-rag (GitHub)](https://github.com/RManLuo/gfm-rag)

---

### Project Overview
This notebook presents an applied validation study of **GFM-RAG** (Graph Foundation Model - Retrieval-Augmented Generation). Modern enterprise RAG pipelines often suffer from hallucination and shallow context retrieval when dealing with complex, multi-hop reasoning over unstructured text corpora. GFM-RAG bridges this gap by automatically extracting semantic entities and relations into a structured Knowledge Graph (KG) using foundation LLMs, then indexing and navigating the graph via Graph Convolutional architectures (e.g., GraphSAGE) for structure-aware retrieval.

### Key Objectives
1. End-to-end pipeline setup and execution of the **Stage 1 Knowledge Graph Construction Workflow** on the HotpotQA multi-hop question-answering corpus.
2. Analysis of the OpenIE triplet extraction and entity-document mapping schema ().
3. Engineering evaluation of graph-augmented retrieval trade-offs and runtime dependencies in production environments.


gfmrag is available via pip

In [1]:
%pip install gfmrag

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 6.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of llama-index-cli to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.2/146.2 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
import gfmrag

/usr/local/lib/python3.12/dist-packages/gfmrag/kg_construction/entity_linking_model/colbert_el_model.py:5: UserWarning: 
********************************************************************************
RAGatouille WARNING: Future Release Notice
--------------------------------------------
RAGatouille version 0.0.10 will be migrating to a PyLate backend 
instead of the current Stanford ColBERT backend.
PyLate is a fully mature, feature-equivalent backend, that greatly facilitates compatibility.
However, please pin version <0.0.10 if you require the Stanford ColBERT backend.
********************************************************************************
  from ragatouille import RAGPretrainedModel


Datasets for gfmrag must conform to a specific folder layout.
For this to work in Colab the shared directory created here (containing corpus and output artefatcts) will be symlinked from the default data directory for a convenient demonstration of the knowledge graph index builder known as the "stage1" workflow.

In [3]:
import os
# Set working directory for GFM-RAG artifacts
project_root = "/content/gfmrag-root"
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    project_root = "/content/drive/MyDrive/gfmrag-root"
except Exception as e:
    print("Running in local or non-mounted environment; using local scratch directory.")

os.makedirs(project_root, exist_ok=True)
print(f"Project root established at: {project_root}")
os.chdir(project_root)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root directory created or already exists at: /content/drive/MyDrive/gfmrag-root


In [ ]:
#!rm /usr/local/lib/python3.12/dist-packages/data
!ln -s /usr/local/lib/python3.12/dist-packages/data /content/drive/MyDrive/gfmrag-root/



One of many yaml files that contain defaults that can be overidden by shell arguments.

I focus on this "workflow" which generates a knowledge graph from a raw corpus.

Choice of ontology detection method via ner_model.
LLM services and text encoding are specified here defaulting to openAI (requires an API key) and Colbert respectively.

local Ollama servers can also participate in KG manufacture.

OpenAI access is tightly throttled so KG triplet generation can be slow.

In [67]:
!cat /usr/local/lib/python3.12/dist-packages/gfmrag/workflow/config/stage1_index_dataset.yaml

hydra:
  run:
    dir: outputs/kg_construction/${now:%Y-%m-%d}/${now:%H-%M-%S} # Output directory

defaults:
  - _self_
  - ner_model: llm_ner_model # The NER model to use
  - openie_model: llm_openie_model # The OpenIE model to use
  - el_model: colbert_el_model # The EL model to use

dataset:
  root:  ./data # data root directory
  data_name: hotpotqa_test # data name

kg_constructor:
  _target_: gfmrag.kg_construction.KGConstructor # The KGConstructor class
  open_ie_model: ${openie_model}
  ner_model: ${ner_model}
  el_model: ${el_model}
  root: tmp/kg_construction # Temporary directory for storing intermediate files during KG construction
  num_processes: 10 # Number of processes to use
  cosine_sim_edges: True # Whether to conduct entities resolution using cosine similarity
  threshold: 0.8 # Threshold for cosine similarity
  max_sim_neighbors: 100 # Maximum number of similar neighbors to add
  add_title: True # Whether to add the title to the content of the document during OpenI

A small sample of the corpus...

In [61]:
!head data/hotpotqa_test/raw/dataset_corpus.json


{
	"Fred Gehrke": "Clarence Fred Gehrke (April 24, 1918 \u2013 February 9, 2002) was an American football player and executive.  He played in the National Football League (NFL) for the Cleveland / Los Angeles Rams, San Francisco 49ers and Chicago Cardinals from 1940 through 1950.  To boost team morale, Gehrke designed and painted the Los Angeles Rams logo in 1948, which was the first painted on the helmets of an NFL team.  He later served as the general manager of the Denver Broncos from 1977 through 1981.  He is the great-grandfather of Miami Marlin Christian Yelich",
	"Manny Machado": "Manuel Arturo Machado (] ; born July 6, 1992) is an American professional baseball third baseman and shortstop for the Baltimore Orioles of Major League Baseball (MLB).  He attended Brito High School in Miami and was drafted by the Orioles with the third overall pick in the 2010 Major League Baseball draft.  He bats and throws right-handed.",
	"Christian Yelich": "Christian Stephen Yelich (born Decembe

The default llm (OpenAI gpt-4o-mini) is used to generate the knowledge graph. The foundation model can be specified in it's own yaml: llm_openie_model.yaml

In [51]:
# Securely set OpenAI API Key for OpenIE extraction
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    if "OPENAI_API_KEY" not in os.environ:
        import getpass
        os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key: ")

# Execute Stage 1 Knowledge Graph generation pipeline
!python -m gfmrag.workflow.stage1_index_dataset kg_constructor.num_processes=5


Ouput is placed in a sibling folder to the corpus and consists of three files.

In [54]:
!ls data/hotpotqa_test/processed/stage1/


document2entities.json	kg.txt	test.json


A sample of the Knowledge Graph produced by the "Stage1" workflow.

In [66]:
!head data/hotpotqa_test/processed/stage1/kg.txt

clarence fred gehrke,was born on,april 24  1918
clarence fred gehrke,died on,february 9  2002
clarence fred gehrke,was,american football player
clarence fred gehrke,played in,nfl
clarence fred gehrke,played for,cleveland rams
clarence fred gehrke,played for,los angeles rams
clarence fred gehrke,played for,san francisco 49ers
clarence fred gehrke,played for,chicago cardinals
clarence fred gehrke,played from,1940
clarence fred gehrke,played until,1950


The accompanying "document to entities" file is an important feature for the classification tasks that allow GFM-RAG to analyse seperate graphs and align them appropriately to formulate sensible facts. But this is beyond the current scope.

In [65]:
!head data/hotpotqa_test/processed/stage1/document2entities.json

{
    "Fred Gehrke": [
        "los angeles rams",
        "general manager",
        "denver broncos",
        "miami marlin",
        "1977",
        "christian yelich",
        "nfl",
        "clarence fred gehrke",


#Summary
GFM-RAG compilation is not for the faint of heart. Compiling from source is frought with GPU related complications revolving around faiss-gpu-cu12 dependency. To validate the full API would be a monumental task requiring a dedicated team to oversee the adaptation to Sovereign Integrity Systems.

But at the very least, the knowledge graph generation has now been validated and this opens the door to testing and implementing all graph related functionalities.

Remaining workflows are provided (stages) by the suite to manage graph retrieval and reasoning and much of their performance characterisitics are determined not only by Stage1 knowledge graph parameters but by pre-training, foundation choice, graph convolution architecture amongst several sets of hyperparameters and architectural choices.



